# 0417_best — submission_lgbm_v2.csv 再現ノートブック

**LBスコア**: 18.092（2026-04-17 時点ベスト）

## パイプライン概要

```
train.csv (raw)
  └─ ベイスギを除外（1240サンプル → 1167サンプル）
      └─ SNV（散乱補正）
          └─ PCA 50次元に圧縮
              └─ 樹種番号を連結（計51次元）
                  └─ LightGBM × GroupKFold(樹種単位, 5fold)
                      └─ 各foldの予測を平均 → submission_lgbm_v2.csv
```

## このコンペの特殊性

- **train の樹種（13種）と test の樹種（6種）が完全に異なる**
- → モデルは「見たことのない樹種」に対して含水率を予測しなければならない（ゼロショット汎化問題）
- → CVも「樹種単位」で分割し、未知樹種への汎化性能を正しく測定する必要がある


## 1. ライブラリ・データ読み込み

### データ形式

| カラム | 内容 |
|---|---|
| `sample number` | サンプル通し番号 |
| `species number` | 樹種番号（整数） |
| `樹種` | 樹種名（文字列） |
| `含水率` | 目的変数（trainのみ）|
| `9993.76781` 〜 `3999.82139` | NIRスペクトル値（吸光度）。約1555列 |

ファイルは **Shift-JIS** エンコーディングのため `encoding='shift-jis'` を指定する。  
スペクトル列の特定は「メタ列でない列」をすべて拾うことで行う。


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

train = pd.read_csv('../data/raw/train.csv', encoding='shift-jis')
test  = pd.read_csv('../data/raw/test.csv',  encoding='shift-jis')

META_COLS      = ['sample number', 'species number', '樹種', '含水率']
WAVENUMBER_COLS = [c for c in train.columns if c not in META_COLS]
TARGET         = '含水率'

print(f'train: {train.shape}, test: {test.shape}')
print(f'波数列数: {len(WAVENUMBER_COLS)}')

train: (1322, 1559), test: (550, 1558)
波数列数: 1555


## 2. 前処理：SNV（Standard Normal Variate）

### なぜ前処理が必要か

NIRスペクトルには以下の測定ノイズが乗る。

- **散乱**：木材表面の粒径・面（板目/まさ目/追いまさ）の違いによる光の散乱量の差
- **光路長変動**：プローブ〜試料間距離がわずかに変わると全波数が一律に上下する
- **ベースラインシフト**：これらの重なりでスペクトル全体がオフセット・傾きを持つ

これらは「含水率とは無関係な変動」なので、除去してから学習したい。

### SNV の仕組み

各サンプル（行）を独立に標準化する。

$$x'_i = \frac{x_i - \bar{x}}{\sigma_x}$$

- $x_i$：そのサンプルの波数 $i$ における吸光度
- $\bar{x}$：そのサンプルの全波数における平均吸光度
- $\sigma_x$：そのサンプルの全波数における標準偏差

**行単位**の処理なので、testスペクトルにも trainの情報を一切使わず独立に適用できる。  
→ ルール「未知スペクトルが1つだけ与えられた状況でも実行可能か？」を満たす。


In [2]:
def snv(X):
    """Standard Normal Variate: 行単位で (X - mean) / std"""
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    return (X - mean) / (std + 1e-10)

X_test_raw = test[WAVENUMBER_COLS].values.astype(float)
X_test_snv = snv(X_test_raw)

print('SNV 完了')

SNV 完了


## 3. ベイスギ除外 & 特徴量構築（PCA + 樹種番号）

### ベイスギを除外する理由

EDAにより、ベイスギはスペクトルの挙動が他樹種と大きく異なることが判明している。  
trainに含めると「ベイスギに合わせた」モデルになり、testの6樹種への汎化が悪化する可能性がある。  
除外後：1240サンプル（13樹種） → 1167サンプル（12樹種）

### PCAによる次元削減

SNV後のスペクトルは約1555次元。LightGBMにそのまま入れると：

- 学習が不安定になる（次元の呪い）
- 隣接する波数は高度に相関しており冗長

PCA（主成分分析）で情報を圧縮する。

- **n_components=50**：50次元に削減（元の約3%の次元数）
- **重要**：PCAは `train_v2`（ベイスギ除外済み）のみで `fit` する  
  testには `transform` のみを適用する（ルール遵守・データリーク防止）

### 最終特徴量

| 特徴量 | 次元数 | 説明 |
|---|---|---|
| PCA主成分 | 50 | SNV済みスペクトルを圧縮した潜在表現 |
| 樹種番号 | 1 | `species number`（整数）。樹種ごとの固有特性を伝える |
| **合計** | **51** | |

> **注意**：testの6樹種はtrainに登場しない樹種番号を持つため、  
> 「樹種番号」はtestに対してほぼ意味をなさない。  
> スペクトルのPCA成分が汎化の主体になっている。


In [3]:
# ベイスギを除外
train_v2 = train[train['樹種'] != 'ベイスギ'].reset_index(drop=True)
print(f'除外前: {len(train)} → 除外後: {len(train_v2)} サンプル')
print(f'残り樹種数: {train_v2["樹種"].nunique()}')

X_train_raw_v2 = train_v2[WAVENUMBER_COLS].values.astype(float)
y_train_v2     = train_v2[TARGET].values

X_train_snv_v2 = snv(X_train_raw_v2)

# PCA: train_v2 でのみ fit
pca_v2         = PCA(n_components=50, random_state=42)
X_train_pca_v2 = pca_v2.fit_transform(X_train_snv_v2)
X_test_pca_v2  = pca_v2.transform(X_test_snv)

# PCA50次元 + 樹種番号
X_feat_v2      = np.hstack([X_train_pca_v2, train_v2['species number'].values.reshape(-1, 1)])
X_feat_test_v2 = np.hstack([X_test_pca_v2,  test['species number'].values.reshape(-1, 1)])

print(f'学習特徴量: {X_feat_v2.shape}, テスト特徴量: {X_feat_test_v2.shape}')

除外前: 1322 → 除外後: 1210 サンプル
残り樹種数: 12
学習特徴量: (1210, 51), テスト特徴量: (550, 51)


## 4. LightGBM 学習（GroupKFold / 樹種単位）

### なぜ GroupKFold を樹種単位にするのか

testの樹種（クスノキ・ケヤキ・スギ・タモ・チーク・ヤマザクラ）は  
trainに**一切登場しない**。モデルは未知樹種に対して汎化しなければならない。

もし sample number 単位でfoldを分けると：
- 同一樹種のデータがtrain/valに混在
- 「その樹種を見た上で予測する」という楽観的な評価になる
- 実際のtest（全樹種が未知）とのギャップが大きい

樹種単位でfoldを分けると：
- valの樹種はtrainに一切登場しない（本番に近い評価）
- より厳しく・正直なCVになる

### LightGBM パラメータの意味

| パラメータ | 値 | 意味 |
|---|---|---|
| `objective` | `regression` | 回帰タスク |
| `metric` | `rmse` | 評価指標 |
| `learning_rate` | 0.05 | 1ステップの学習幅。小さいほど丁寧だが木の数が増える |
| `num_leaves` | 63 | 1本の木の葉の最大数。多いほど複雑なモデル |
| `min_child_samples` | 10 | 葉に最低限必要なサンプル数。過学習防止 |
| `feature_fraction` | 0.8 | 各木で使う特徴量の割合（ランダムサブサンプリング） |
| `bagging_fraction` | 0.8 | 各木で使うサンプルの割合（行のサブサンプリング） |
| `bagging_freq` | 5 | bagging を何ステップごとに行うか |
| `reg_alpha/lambda` | 0.1 | L1/L2正則化。過学習を抑制 |

### 学習ループの仕組み

```
for 各fold:
  train部分でLightGBMをfit
  val部分でOOF予測を生成（→ CV精度評価に使う）
  test全体を予測して 1/5 を積み上げる

最終test予測 = 5foldの予測の平均
```

**Early stopping（100rounds）**：valのRMSEが100ステップ改善しなければ学習打ち切り。  
過学習を防ぎつつ `num_boost_round=2000` という上限を設けている。

**OOF（Out-of-Fold）予測**：各サンプルは自分が含まれていないfoldの  
モデルで予測される。全サンプル分のOOF予測が揃うと「testなし」でモデル精度を評価できる。


In [4]:
lgb_params = {
    'objective':        'regression',
    'metric':           'rmse',
    'learning_rate':    0.05,
    'num_leaves':       63,
    'min_child_samples': 10,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'verbose':          -1,
    'random_state':     42,
}

groups_v2     = train_v2['species number'].values
kf_v2         = GroupKFold(n_splits=5)
oof_preds_v2  = np.zeros(len(X_feat_v2))
test_preds_v2 = np.zeros(len(X_feat_test_v2))
fold_rmses_v2 = []

for fold, (tr_idx, val_idx) in enumerate(kf_v2.split(X_feat_v2, y_train_v2, groups_v2)):
    val_species = train_v2.iloc[val_idx]['樹種'].unique().tolist()
    print(f'Fold {fold+1} val: {val_species}')

    dtrain = lgb.Dataset(X_feat_v2[tr_idx],  label=y_train_v2[tr_idx])
    dval   = lgb.Dataset(X_feat_v2[val_idx], label=y_train_v2[val_idx], reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)],
    )

    val_pred                = model.predict(X_feat_v2[val_idx])
    oof_preds_v2[val_idx]  = val_pred
    test_preds_v2          += model.predict(X_feat_test_v2) / 5

    rmse = np.sqrt(mean_squared_error(y_train_v2[val_idx], val_pred))
    fold_rmses_v2.append(rmse)
    print(f'  → RMSE = {rmse:.4f}\n')

overall_rmse_v2 = np.sqrt(mean_squared_error(y_train_v2, oof_preds_v2))
print(f'=== OOF RMSE (v2): {overall_rmse_v2:.4f} ===')
print(f'fold平均: {np.mean(fold_rmses_v2):.4f} ± {np.std(fold_rmses_v2):.4f}')

Fold 1 val: ['ウエンジ', 'トチ']
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[21]	valid_0's rmse: 26.8597
  → RMSE = 26.8597

Fold 2 val: ['チェリー', 'ヒノキ']
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[203]	valid_0's rmse: 25.6978
  → RMSE = 25.6978

Fold 3 val: ['ウォールナット', 'クリ']
Training until validation scores don't improve for 100 rounds
[500]	valid_0's rmse: 19.8865
[1000]	valid_0's rmse: 19.8281
Early stopping, best iteration is:
[944]	valid_0's rmse: 19.8275
  → RMSE = 19.8275

Fold 4 val: ['ナラ', 'ベイマツ', 'ホワイトオーク']
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[41]	valid_0's rmse: 9.83036
  → RMSE = 9.8304

Fold 5 val: ['イチョウ', 'スプルース', '米ヒバ']
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[104]	valid_0's rmse: 23.6339
  → RMSE = 23.6339

=== OOF RMSE (v2): 22.1557 ===
fold平均: 21.1699 

## 5. CV RMSE 確認（改善の判断基準）

### 運用ルール

- **毎回確認するのはここの OOF RMSE**。これを下げることを最優先に改善する。
- LBスコアへの投稿は1日に回数制限があるため、CV RMSEが改善したと判断したときだけ Section 6 を実行する。
- CV RMSE が改善しても LB が悪化するケースもあるため、両方の履歴を手元に記録しておく。

### スコア履歴

| 日付 | バージョン | OOF RMSE | LBスコア | メモ |
|---|---|---|---|---|
| 2026-04-17 | v2（ベースライン） | 22.1557 | 18.092 | SNV + PCA50 + LightGBM GroupKFold |


In [5]:
# ===== CV RMSE サマリー =====
print('=' * 45)
print(f'OOF RMSE : {overall_rmse_v2:.4f}')
print(f'fold平均  : {np.mean(fold_rmses_v2):.4f} ± {np.std(fold_rmses_v2):.4f}')
print('-' * 45)
for i, r in enumerate(fold_rmses_v2, 1):
    print(f'  Fold {i}: {r:.4f}')
print('=' * 45)
print()
print('▼ ベースラインとの比較')
baseline_rmse = 22.1557
diff = overall_rmse_v2 - baseline_rmse
sign = '+' if diff >= 0 else ''
print(f'  ベースライン (v2): {baseline_rmse:.4f}')
print(f'  今回             : {overall_rmse_v2:.4f}  ({sign}{diff:.4f})')
print()
if diff < 0:
    print('  → 改善！Section 6 の投稿コードの実行を検討する。')
elif diff == 0:
    print('  → 変化なし。')
else:
    print('  → 悪化。Section 6 は実行しない。')

OOF RMSE : 22.1557
fold平均  : 21.1699 ± 6.1536
---------------------------------------------
  Fold 1: 26.8597
  Fold 2: 25.6978
  Fold 3: 19.8275
  Fold 4: 9.8304
  Fold 5: 23.6339

▼ ベースラインとの比較
  ベースライン (v2): 22.1557
  今回             : 22.1557  (+0.0000)

  → 悪化。Section 6 は実行しない。


## 6. Submission 保存（CV RMSE 改善を確認してから実行）

> **デフォルトはコメントアウト。**  
> 1日の投稿回数に上限があるため、Section 5 で OOF RMSE がベースラインより改善したと判断した場合のみ  
> 下のセルのコメントを外して実行する。

### 提出フォーマットの注意点

- **1列目**：`sample number`（テストデータのID）
- **2列目**：予測した含水率（%）
- `header=False`：ヘッダー行なし
- `encoding='shift-jis'`：日本語文字コード


In [6]:
# ===== 投稿時のみコメントを外して実行 =====
# CV RMSE がベースライン (22.1557) より改善したことを確認してから実行すること。

# sub_v2 = pd.DataFrame({
#     0: test['sample number'].values,
#     1: test_preds_v2,
# })
# sub_v2.to_csv('../data/processed/submission_lgbm_v2.csv',
#               index=False, header=False, encoding='shift-jis')
#
# print('保存: data/processed/submission_lgbm_v2.csv')
# print(f'予測値 range: {test_preds_v2.min():.2f} ~ {test_preds_v2.max():.2f} %')
# print(sub_v2.head(10))